In [7]:
from __future__ import annotations

import gc
import json
import os
import random
import re
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import FeatureUnion, Pipeline
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup


In [8]:
@dataclass
class CFG:
    seed: int = 42
    input_dir: str = "/kaggle/input/competitions/nlp-getting-started"
    output_dir: str = "/kaggle/working"
    model_name: str = "roberta-base"
    n_splits: int = 5
    max_length: int = 128
    epochs: int = 4
    train_batch_size: int = 16
    valid_batch_size: int = 32
    learning_rate: float = 1e-5
    weight_decay: float = 0.01
    warmup_ratio: float = 0.1
    use_keyword: bool = True
    use_location: bool = False

In [9]:
def seed_everything(seed: int) -> None:
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False




def is_missing(value) -> bool:
    return value is None or value != value or str(value).strip() == ""


def clean_text(text: str) -> str:
    text = str(text)
    text = URL_RE.sub(" URL ", text)
    text = MENTION_RE.sub(" USER ", text)
    text = HTML_RE.sub(" ", text)
    text = text.replace("#", " ")
    text = SPACE_RE.sub(" ", text)
    return text.strip()

In [10]:

def build_text(row: dict) -> str:
    pieces = []
    if cfg.use_keyword and not is_missing(row.get("keyword")):
        pieces.append(f"keyword: {row['keyword']}")
    if cfg.use_location and not is_missing(row.get("location")):
        pieces.append(f"location: {row['location']}")
    pieces.append(str(row.get("text", "")))
    return clean_text(" ".join(pieces))



class TweetDataset(Dataset):
    def __init__(self, texts, labels=None):
        self.texts = list(texts)
        self.labels = None if labels is None else list(labels)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        item = {"text": self.texts[idx]}
        if self.labels is not None:
            item["label"] = int(self.labels[idx])
        return item


def collate_fn(batch, tokenizer, max_length):
    texts = [item["text"] for item in batch]
    enc = tokenizer(texts, padding=True, truncation=True, max_length=max_length, return_tensors="pt")
    if "label" in batch[0]:
        enc["labels"] = torch.tensor([item["label"] for item in batch], dtype=torch.long)
    return enc


In [11]:

cfg = CFG()


URL_RE = re.compile(r"https?://\S+|www\.\S+")
HTML_RE = re.compile(r"&[a-z]+;")
MENTION_RE = re.compile(r"@\w+")
SPACE_RE = re.compile(r"\s+")

In [12]:

def metric_dict(y_true, y_pred, y_score=None) -> dict[str, float]:
    out = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
    }
    if y_score is not None and len(np.unique(y_true)) == 2:
        y_score = np.nan_to_num(y_score, nan=0.5, posinf=1.0, neginf=0.0)
        out["roc_auc"] = roc_auc_score(y_true, y_score)
    return {k: float(v) for k, v in out.items()}


def find_best_threshold(y_true, scores, start=0.0, stop=1.0, steps=401) -> tuple[float, float]:
    scores = np.nan_to_num(scores, nan=0.5, posinf=1.0, neginf=0.0)
    best_threshold = 0.5
    best_f1 = -1.0
    for threshold in np.linspace(start, stop, steps):
        preds = (scores >= threshold).astype(int)
        score = f1_score(y_true, preds, zero_division=0)
        if score > best_f1:
            best_threshold = float(threshold)
            best_f1 = float(score)
    return best_threshold, best_f1


In [13]:
def make_tfidf_model() -> Pipeline:
    return Pipeline(
        [
            (
                "features",
                FeatureUnion(
                    [
                        (
                            "word",
                            TfidfVectorizer(
                                analyzer="word",
                                max_features=60000,
                                ngram_range=(1, 2),
                                min_df=2,
                                sublinear_tf=True,
                            ),
                        ),
                        (
                            "char",
                            TfidfVectorizer(
                                analyzer="char_wb",
                                ngram_range=(3, 5),
                                min_df=2,
                                sublinear_tf=True,
                            ),
                        ),
                    ]
                ),
            ),
            ("clf", LogisticRegression(C=2.0, max_iter=2000, solver="liblinear", random_state=cfg.seed)),
        ]
    )


def predict_transformer_scores(model, tokenizer, texts, device) -> np.ndarray:
    loader = DataLoader(
        TweetDataset(texts),
        batch_size=cfg.valid_batch_size,
        shuffle=False,
        collate_fn=lambda batch: collate_fn(batch, tokenizer, cfg.max_length),
    )
    model.eval()
    scores = []
    with torch.no_grad():
        for batch in tqdm(loader, desc="predict", leave=False):
            batch = {k: v.to(device) for k, v in batch.items()}
            logits = model(**batch).logits
            probs = torch.softmax(logits, dim=-1)[:, 1]
            scores.extend(probs.detach().cpu().numpy().tolist())
    return np.nan_to_num(np.array(scores), nan=0.5, posinf=1.0, neginf=0.0)


In [15]:
def train_transformer_fold(train_df, valid_df, fold: int):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    tokenizer = AutoTokenizer.from_pretrained(cfg.model_name)
    model = AutoModelForSequenceClassification.from_pretrained(cfg.model_name, num_labels=2).to(device)

    train_loader = DataLoader(
        TweetDataset(train_df["clean_text"], train_df["target"]),
        batch_size=cfg.train_batch_size,
        shuffle=True,
        collate_fn=lambda batch: collate_fn(batch, tokenizer, cfg.max_length),
    )
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.learning_rate, weight_decay=cfg.weight_decay)
    total_steps = len(train_loader) * cfg.epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(total_steps * cfg.warmup_ratio),
        num_training_steps=total_steps,
    )

    best = {"f1": -1.0, "state": None, "valid_scores": None, "threshold": 0.5}
    y_valid = valid_df["target"].values
    for epoch in range(cfg.epochs):
        model.train()
        losses = []
        progress = tqdm(train_loader, desc=f"fold {fold} epoch {epoch + 1}/{cfg.epochs}")
        for batch in progress:
            batch = {k: v.to(device) for k, v in batch.items()}
            optimizer.zero_grad(set_to_none=True)
            out = model(**batch)
            if not torch.isfinite(out.loss):
                raise RuntimeError("NaN loss. Switch model or lower learning_rate.")
            out.loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            losses.append(out.loss.item())
            progress.set_postfix(loss=np.mean(losses[-50:]))

        valid_scores = predict_transformer_scores(model, tokenizer, valid_df["clean_text"].tolist(), device)
        threshold, valid_f1 = find_best_threshold(y_valid, valid_scores)
        valid_preds = (valid_scores >= threshold).astype(int)
        print(
            f"fold {fold} epoch {epoch + 1}:",
            json.dumps(metric_dict(y_valid, valid_preds, valid_scores), indent=2),
            "threshold:",
            threshold,
        )
        if valid_f1 > best["f1"]:
            best = {
                "f1": valid_f1,
                "state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
                "valid_scores": valid_scores,
                "threshold": threshold,
            }

    model.load_state_dict(best["state"])
    return model, tokenizer, best



def normalize_for_match(text: str) -> str:
    text = str(text)
    text = re.sub(r"http\S+|www\.\S+", " URL ", text)
    text = re.sub(r"@\w+", " USER ", text)
    text = re.sub(r"&[a-z]+;", " ", text)
    text = text.replace("#", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip().lower()

In [16]:
seed_everything(cfg.seed)
input_dir = Path(cfg.input_dir)
output_dir = Path(cfg.output_dir)

train = pd.read_csv(input_dir / "train.csv")
test = pd.read_csv(input_dir / "test.csv")

train["clean_text"] = [build_text(row) for row in train.to_dict(orient="records")]
test["clean_text"] = [build_text(row) for row in test.to_dict(orient="records")]

y = train["target"].values
oof_tfidf = np.zeros(len(train))
oof_transformer = np.zeros(len(train))
test_tfidf = np.zeros(len(test))
test_transformer = np.zeros(len(test))
fold_metrics = []

skf = StratifiedKFold(n_splits=cfg.n_splits, shuffle=True, random_state=cfg.seed)
for fold, (train_idx, valid_idx) in enumerate(skf.split(train, y), start=1):
    print(f"\n===== Fold {fold}/{cfg.n_splits} =====")
    train_df = train.iloc[train_idx].reset_index(drop=True)
    valid_df = train.iloc[valid_idx].reset_index(drop=True)

    tfidf = make_tfidf_model()
    tfidf.fit(train_df["clean_text"], train_df["target"])
    oof_tfidf[valid_idx] = tfidf.predict_proba(valid_df["clean_text"])[:, 1]
    test_tfidf += tfidf.predict_proba(test["clean_text"])[:, 1] / cfg.n_splits

    model, tokenizer, best = train_transformer_fold(train_df, valid_df, fold)
    oof_transformer[valid_idx] = best["valid_scores"]
    test_transformer += predict_transformer_scores(
        model,
        tokenizer,
        test["clean_text"].tolist(),
        torch.device("cuda" if torch.cuda.is_available() else "cpu"),
    ) / cfg.n_splits
    fold_metrics.append({"fold": fold, "transformer_f1": best["f1"], "threshold": best["threshold"]})

    del model, tokenizer, tfidf
    gc.collect()
    torch.cuda.empty_cache()

best_final = None
for alpha in np.linspace(0, 1, 41):
    scores = alpha * oof_transformer + (1 - alpha) * oof_tfidf
    threshold, _ = find_best_threshold(y, scores)
    preds = (scores >= threshold).astype(int)
    result = metric_dict(y, preds, scores)
    result["alpha_transformer"] = float(alpha)
    result["threshold"] = threshold
    if best_final is None or result["f1"] > best_final["f1"]:
        best_final = result

print("\nFold metrics:", json.dumps(fold_metrics, indent=2))
print("Best OOF ensemble:", json.dumps(best_final, indent=2))

final_scores = best_final["alpha_transformer"] * test_transformer + (1 - best_final["alpha_transformer"]) * test_tfidf
final_preds = (final_scores >= best_final["threshold"]).astype(int)



train_norm = train["text"].apply(normalize_for_match)
test_norm = test["text"].apply(normalize_for_match)

train_labels_by_text = train.groupby(train_norm)["target"].agg(lambda x: list(set(x))).to_dict()

unique_overlap_indices = []
true_labels = []
for idx, txt in enumerate(test_norm):
    labels = train_labels_by_text.get(txt, [])
    if len(labels) == 1:
        unique_overlap_indices.append(idx)
        true_labels.append(labels[0])

final_preds_corrected = final_preds.copy()
for idx, lbl in zip(unique_overlap_indices, true_labels):
    final_preds_corrected[idx] = lbl

print(f"\nИсправлено предсказаний для {len(unique_overlap_indices)} тестовых строк (уникальное совпадение с train).")

submission = pd.DataFrame({"id": test["id"], "target": final_preds_corrected})
submission.to_csv(output_dir / "submission.csv", index=False)
np.save(output_dir / "test_scores.npy", final_scores)
with (output_dir / "metrics.json").open("w", encoding="utf-8") as file:
    json.dump(
        {
            "model_name": cfg.model_name,
            "n_splits": cfg.n_splits,
            "fold_metrics": fold_metrics,
            "best_oof_ensemble": best_final,
            "num_overlap_corrected": len(unique_overlap_indices),
        },
        file,
        indent=2,
    )
print("Saved submission:", output_dir / "submission.csv")
print(submission.head())



===== Fold 1/5 =====


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


fold 1 epoch 1/4:   0%|          | 0/381 [00:00<?, ?it/s]

predict:   0%|          | 0/48 [00:00<?, ?it/s]

...

fold 5 epoch 4/4:   0%|          | 0/381 [00:00<?, ?it/s]

predict:   0%|          | 0/48 [00:00<?, ?it/s]

fold 5 epoch 4: {
  "accuracy": 0.842969776609724,
  "precision": 0.8187403993855606,
  "recall": 0.8149847094801224,
  "f1": 0.8168582375478928,
  "roc_auc": 0.8987020673910286
} threshold: 0.2925


predict:   0%|          | 0/102 [00:00<?, ?it/s]


Fold metrics: [
  {
    "fold": 1,
    "transformer_f1": 0.8205128205128205,
    "threshold": 0.7875
  },
  {
    "fold": 2,
    "transformer_f1": 0.8122481869460113,
    "threshold": 0.5475
  },
  {
    "fold": 3,
    "transformer_f1": 0.8056206088992974,
    "threshold": 0.6025
  },
  {
    "fold": 4,
    "transformer_f1": 0.805877803557618,
    "threshold": 0.425
  },
  {
    "fold": 5,
    "transformer_f1": 0.8168582375478928,
    "threshold": 0.2925
  }
]
Best OOF ensemble: {
  "accuracy": 0.8409299881781164,
  "precision": 0.8419654714475432,
  "recall": 0.7752980739834913,
  "f1": 0.8072576794524908,
  "roc_auc": 0.8966488512521791,
  "alpha_transformer": 0.7000000000000001,
  "threshold": 0.5675
}

Исправлено предсказаний для 333 тестовых строк (уникальное совпадение с train).
Saved submission: /kaggle/working/submission.csv
   id  target
0   0       1
1   2       1
2   3       1
3   9       1
4  11       1
